In [3]:
url_base = "https://g1.globo.com/sitemap/g1/sitemap.xml"

In [4]:
import requests
from bs4 import BeautifulSoup

sitemap_response = requests.get(url_base)
sitemap_content = sitemap_response.content
soup = BeautifulSoup(sitemap_content, 'xml')

In [5]:
urls_dates = []

for item in soup.find_all('sitemap'):
    loc = item.find('loc').text
    lastmod = item.find('lastmod').text
    urls_dates.append((loc, lastmod))

In [6]:
import pandas as pd

df = pd.DataFrame(urls_dates, columns=['URL', 'Last Modified'])
df.head()

,URL,Last Modified
0,https://g1.globo.com/sitemap/g1/2026/03/25_1.xml,2026-03-25
1,https://g1.globo.com/sitemap/g1/2026/03/24_1.xml,2026-03-24
2,https://g1.globo.com/sitemap/g1/2026/03/23_1.xml,2026-03-23
3,https://g1.globo.com/sitemap/g1/2026/03/22_1.xml,2026-03-22
4,https://g1.globo.com/sitemap/g1/2026/03/21_1.xml,2026-03-21


In [7]:
# keep all the ones from a specific date range
df['Last Modified'] = pd.to_datetime(df['Last Modified'])
filtered_df = df[(df['Last Modified'] >= '2020-03-25') & (df['Last Modified'] <= '2025-03-25')]
print(filtered_df.shape)
filtered_df.head()

(2241, 2)


,URL,Last Modified
365,https://g1.globo.com/sitemap/g1/2025/03/25_1.xml,2025-03-25
366,https://g1.globo.com/sitemap/g1/2025/03/24_1.xml,2025-03-24
367,https://g1.globo.com/sitemap/g1/2025/03/23_1.xml,2025-03-23
368,https://g1.globo.com/sitemap/g1/2025/03/22_1.xml,2025-03-22
369,https://g1.globo.com/sitemap/g1/2025/03/21_1.xml,2025-03-21


In [8]:
url_sitemap = filtered_df['URL'].iloc[0]
print(url_sitemap)

https://g1.globo.com/sitemap/g1/2025/03/25_1.xml


In [9]:
sitemap_response = requests.get(url_sitemap)
sitemap_content = sitemap_response.content
soup = BeautifulSoup(sitemap_content, 'xml')

In [10]:
urls = soup.find_all('loc')

filtered_urls = [url.text for url in urls if 'fato-ou-fake' in url.text]
print(len(filtered_urls))
filtered_urls

3


['https://g1.globo.com/fato-ou-fake/noticia/2025/03/25/e-fake-video-em-que-cesar-tralli-recomenda-link-para-resgatar-r-7-mil-usando-cpf-trata-se-de-golpe.ghtml',
 'https://g1.globo.com/fato-ou-fake/noticia/2025/03/25/e-fake-que-o-papa-francisco-chama-a-todos-hoje-as-21h-para-oracao-pela-paz-na-siria.ghtml',
 'https://g1.globo.com/fato-ou-fake/noticia/2025/03/25/e-fake-video-que-mostra-navio-de-cruzeiro-despejando-enxurrada-de-esgoto-no-oceano.ghtml']

## Getting all fato ou fake

In [11]:
import csv

run = False
if run:
    for index, row in filtered_df.iterrows():
        url_sitemap = row['URL']
        print(f"Processing sitemap: {url_sitemap}")
        
        sitemap_response = requests.get(url_sitemap)
        sitemap_content = sitemap_response.content
        soup = BeautifulSoup(sitemap_content, 'xml')
        
        urls = soup.find_all('loc')
        
        filtered_urls = [url.text for url in urls if 'fato-ou-fake' in url.text]
        print(f"Found {len(filtered_urls)} URLs in this sitemap.")

        csv_filename = f"fato_ou_fake_urls.csv"
        with open(csv_filename, mode='a', newline='', encoding='utf-8') as csv_file:
            writer = csv.writer(csv_file)
            for url in filtered_urls:
                writer.writerow([url])

    print("All sitemaps processed. URLs saved to fato_ou_fake_urls.csv")

In [12]:
# 1. Batch write to CSV (biggest impact)
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from bs4 import BeautifulSoup

run = False

def fetch_urls_from_sitemap(url_sitemap):
    """Fetch URLs from a single sitemap"""
    try:
        sitemap_response = requests.get(url_sitemap, timeout=10)
        soup = BeautifulSoup(sitemap_response.content, 'xml')
        urls = soup.find_all('loc')
        return [url.text for url in urls if 'fato-ou-fake' in url.text]
    except Exception as e:
        print(f"Error processing {url_sitemap}: {e}")
        return []


if run:
    # Fetch all URLs from all sitemaps in parallel
    all_urls = []
    with ThreadPoolExecutor(max_workers=5) as executor:  # Adjust based on your internet
        futures = {executor.submit(fetch_urls_from_sitemap, url): url 
                for url in filtered_df['URL']}
        for future in as_completed(futures):
            urls = future.result()
            all_urls.extend(urls)
            print(f"Found {len(urls)} URLs")

    # Write ALL URLs to CSV once (batch I/O)
    with open('fato_ou_fake_urls.csv', mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        for url in all_urls:
            writer.writerow([url])

    print(f"Total: {len(all_urls)} URLs saved")

In [46]:
ff1 = pd.read_csv("result/fato_ou_fake_urls-1.csv", header=None, names=['URL'])

ff2 = pd.read_csv("result/fato_ou_fake_urls.csv", header=None, names=['URL'])

dff = pd.concat([ff1, ff2], ignore_index=True)

# remove URLs that end with .jpg
dff = dff[~dff['URL'].str.endswith('.jpg')]

dff.to_csv("result/dff.csv", index=False)

print(dff.shape)
dff.head()

(2913, 1)


,URL
0,https://g1.globo.com/fato-ou-fake/noticia/2026...
1,https://g1.globo.com/fato-ou-fake/noticia/2026...
2,https://g1.globo.com/fato-ou-fake/noticia/2026...
3,https://g1.globo.com/fato-ou-fake/noticia/2026...
4,https://g1.globo.com/fato-ou-fake/noticia/2026...


## Inspecting Site

In [14]:
url = dff['URL'].iloc[-1285]
print(url)

url = "https://g1.globo.com/df/distrito-federal/eleicoes/2022/noticia/2022/09/07/veja-o-que-e-fato-ou-fake-na-entrevista-de-leandro-grass-para-o-g1.ghtml"

https://g1.globo.com/df/distrito-federal/eleicoes/2022/noticia/2022/09/07/veja-o-que-e-fato-ou-fake-na-entrevista-de-leandro-grass-para-o-g1.ghtml


In [ ]:
response = requests.get(url)
html = response.content
soup = BeautifulSoup(html, 'html.parser')
print(soup.title.text)

Veja o que é #FATO ou #FAKE na entrevista de Leandro Grass para o g1 | G1


In [20]:
from seleniumbase import SB
import time
import random


def human_delay(min_s=1.5, max_s=4.0):
    """Random delay to mimic human reading/waiting."""
    time.sleep(random.uniform(min_s, max_s))


def scrape_article(url: str) -> str:
    with SB(
        uc=True,                  # Undetected Chrome mode
        headless=False,           # Visible browser (more human-like; set True for servers)
        locale_code="en-US",
        # Realistic window size (avoid perfect round numbers)
        window_size="1367,769",
    ) as sb:

        # 1. Open the page
        sb.uc_open_with_reconnect(url, reconnect_time=4)

        # 2. Human-like: slow scroll down the page
        #human_delay(2, 4)
        #sb.execute_script("window.scrollTo({ top: 300, behavior: 'smooth' });")
        #human_delay(1, 2)
        #sb.execute_script("window.scrollTo({ top: 700, behavior: 'smooth' });")
        #human_delay(1, 2)
        #sb.execute_script("window.scrollTo({ top: 1200, behavior: 'smooth' });")
        #human_delay(1.5, 3)

        # 3. Scroll back up slightly (humans rarely read perfectly top-to-bottom)
        sb.execute_script("window.scrollTo({ top: 900, behavior: 'smooth' });")
        human_delay(1, 2)

        # 4. Wait for full page load
        sb.wait_for_element("body", timeout=3)

        # 5. Grab and return the full rendered HTML
        html = sb.get_page_source()
        return html

html = scrape_article(url)

with open("article.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Done. Saved {len(html):,} characters to article.html")

Done. Saved 1,190,005 characters to article.html


In [28]:
# try to get main tag from article.html
html = None
with open("article.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, 'html.parser')
main_content = soup.find('main')

print(str(main_content)[:100])

<main class="mc-body theme" itemscope="" itemtype="http://schema.org/NewsArticle"> <link href="https


In [ ]:
# below <div class="mc-article-header">
# - div class == title
# - div class contains subtitle
# - all tags with class contains content-publication-data

# below <div class="mc-article-body">
# - get all <script type="application/ld+json"> (treat whats inside as str, we can treat later)
# - below <article itemprop="articleBody">
#   - get all (<p>, <h1>, <h2>, <ul>, <li>, blockquote) tags (treat as content, keep order)

In [40]:
def parse_g1_article(soup):
    article_data = {
        'title': None,
        'subtitle': None,
        'from_publication': None,
        'date_publication': None,
        'json_ld': None,
        'content': [],
        #'raw_content_tags': []
    }
    
    # Extract header info
    header = soup.find('div', class_='mc-article-header')
    if header:
        # Get title
        title_div = header.find('div', class_='title')
        if title_div:
            article_data['title'] = title_div.get_text(strip=True)
        
        # Get subtitle
        subtitle_div = header.find('div', class_=lambda x: x and 'subtitle' in x if x else False)
        if subtitle_div:
            article_data['subtitle'] = subtitle_div.get_text(strip=True)

        from_pub_data = header.find('p', class_=lambda x: x and 'content-publication-data__from' in x if x else False)
        if from_pub_data:
            article_data['from_publication'] = from_pub_data.get_text(strip=True)

        updated_pub_data = header.find('p', class_=lambda x: x and 'content-publication-data__updated' in x if x else False)
        if updated_pub_data:
            article_data['date_publication'] = updated_pub_data.get_text(strip=True)
    
    # Extract body info
    body = soup.find('div', class_='mc-article-body')
    if body:
        # Extract JSON-LD scripts
        json_ld_scripts = body.find_all('script', type='application/ld+json')
        if json_ld_scripts:
            import json
            article_data['json_ld'] = []
            for script in json_ld_scripts:
                try:
                    article_data['json_ld'].append(json.loads(script.string))
                except json.JSONDecodeError:
                    article_data['json_ld'].append(script.string)
        
        # Extract article body content
        article_body = body.find('article', attrs={'itemprop': 'articleBody'})
        if article_body:
            # Get all content tags in order: p, h1, h2, h3, h4, ul, li, blockquote
            content_tags = article_body.find_all([
                'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
                'ul', 'li', 'ol',
                'blockquote',
                'figcaption',
                'table', 'tr', 'td', 'th',
                'section',
                'pre', 'code',
                'strong', 'em'
            ])
            
            for tag in content_tags:
                tag_data = {
                    'tag': tag.name,
                    'text': tag.get_text(strip=True),
                    'html': str(tag)
                }
                article_data['content'].append(tag_data)
                #article_data['raw_content_tags'].append((tag.name, tag.get_text(strip=True)))
    
    return article_data

# Usage example:
parsed = parse_g1_article(soup)
print(parsed['title'])
print(parsed['subtitle'])
print(parsed['date_publication'])
for item in parsed['content']:
    print(f"{item['tag']}: {item['text']}")

Veja o que é #FATO ou #FAKE na entrevista de Leandro Grass para o g1
Candidato da federação PV, PT e PC do B ao GDF participou da série de entrevistas com postulantes ao Palácio do Buriti. Podcast foi gravado no estúdio do g1, em Brasília, no dia 26 de agosto e exibido nesta terça-feira (6).
07/09/2022 07h28Atualizadohá 3 anos
figcaption: 1 de 3
Leandro Grass, candidato ao GDF pela federação PV, PT e PC do B, é entrevistado pelo g1 — Foto: Reprodução/TV Globo
p: Leandro Grass, candidato ao GDF pela federação PV, PT e PC do B, é entrevistado pelo g1 — Foto: Reprodução/TV Globo
p: O candidato ao governo do Distrito Federal, Leandro Grass, da federaçãoPV,PTePC do B, foi o entrevistado dog1na série com os postulantes ao Palácio do Buriti, nas eleições de outubro. A entrevista foi gravada no estúdio do g1, emBrasília, no dia 26 de agosto e exibida nesta terça-feira (6).
strong: g1
p: A íntegra está no g1 DFe nas plataformas de áudio (ouça podcast abaixo).
strong: ouça podcast abaixo
p: VEJA

In [41]:
parsed

{'title': 'Veja o que é #FATO ou #FAKE na entrevista de Leandro Grass para o g1',
 'subtitle': 'Candidato da federação PV, PT e PC do B ao GDF participou da série de entrevistas com postulantes ao Palácio do Buriti. Podcast foi gravado no estúdio do g1, em Brasília, no dia 26 de agosto e exibido nesta terça-feira (6).',
 'from_publication': 'Por g1 DF e TV Globo',
 'date_publication': '07/09/2022 07h28Atualizadohá 3 anos',
 'json_ld': [{'articleSection': ['G1',
    'DF',
    'Distrito Federal',
    'Eleições',
    'Eleições 2022 no Distrito Federal']}],
 'content': [{'tag': 'figcaption',
   'text': '1 de 3\nLeandro Grass, candidato ao GDF pela federação PV, PT e PC do B, é entrevistado pelo g1 — Foto: Reprodução/TV Globo',
   'html': '<figcaption hidden=""> 1 de 3\nLeandro Grass, candidato ao GDF pela federação PV, PT e PC do B, é entrevistado pelo g1 — Foto: Reprodução/TV Globo </figcaption>'},
  {'tag': 'p',
   'text': 'Leandro Grass, candidato ao GDF pela federação PV, PT e PC do B,

## Do the process for the rest of urls

In [45]:
import asyncio
import json
import os
import zipfile
import glob
import gc
from datetime import datetime
from pathlib import Path
from bs4 import BeautifulSoup
from typing import List, Dict, Optional
import random
from collections import deque

class G1ArticleScraperOptimized:
    def __init__(
        self,
        output_dir: str = "data/scraped_articles",
        html_dir: str = "data/html_cache",
        max_concurrent: int = 3,  # Max scraping tasks at once
        parse_workers: int = 2,   # Parallel parsing tasks
        min_wait: float = 2.0,
        max_wait: float = 5.0,
        zip_interval: int = 100   # Zip every N files
    ):
        self.output_dir = Path(output_dir)
        self.html_dir = Path(html_dir)
        self.max_concurrent = max_concurrent
        self.parse_workers = parse_workers
        self.min_wait = min_wait
        self.max_wait = max_wait
        self.zip_interval = zip_interval
        
        self.html_counter = 0
        self.parsed_counter = 0
        self.failed_counter = 0
        self.batch_counter = 0
        
        self.results_file = self.output_dir / "articles.jsonl"
        self.failed_file = self.output_dir / "failed_urls.jsonl"
        
        # Queue for scraped HTML (URL, HTML, filepath)
        self.scrape_queue: asyncio.Queue = asyncio.Queue()
        
        # Create directories
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.html_dir.mkdir(parents=True, exist_ok=True)
    
    async def scrape_with_delay(self, url: str, semaphore: asyncio.Semaphore) -> Optional[tuple]:
        """Scrape a single URL with rate limiting via semaphore."""
        async with semaphore:
            try:
                delay = random.uniform(self.min_wait, self.max_wait)
                await asyncio.sleep(delay)
                
                loop = asyncio.get_event_loop()
                html = await loop.run_in_executor(None, scrape_article, url)
                
                # Save HTML immediately
                filename = f"{self.html_counter:06d}_article.html"
                filepath = self.html_dir / filename
                
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(html)
                
                self.html_counter += 1
                print(f"✓ Scraped & saved: {url[:50]}... ({len(html)} chars)")
                
                # Zip every N files
                if self.html_counter % self.zip_interval == 0:
                    await self.zip_batch_async(self.batch_counter)
                    self.batch_counter += 1
                
                return (url, html, str(filepath))
                
            except Exception as e:
                print(f"✗ Scrape failed: {url[:50]}... {str(e)[:50]}")
                await self.log_failed_url(url, str(e))
                return None
    
    async def parse_worker(self):
        """Continuously parse HTML from queue and save JSONL."""
        while True:
            try:
                result = await asyncio.wait_for(self.scrape_queue.get(), timeout=5.0)
                
                if result is None:  # Sentinel value to stop
                    break
                
                url, html, filepath = result
                
                try:
                    soup = BeautifulSoup(html, 'html.parser')
                    article_data = parse_g1_article(soup)
                    article_data['url'] = url
                    article_data['html_path'] = filepath
                    article_data['scraped_at'] = datetime.now().isoformat()
                    
                    # Save to JSONL immediately
                    with open(self.results_file, 'a', encoding='utf-8') as f:
                        f.write(json.dumps(article_data, ensure_ascii=False) + '\n')
                    
                    self.parsed_counter += 1
                    print(f"📝 Parsed & saved: {url[:50]}... ({len(article_data['content'])} content blocks)")
                    
                except Exception as e:
                    print(f"✗ Parse failed: {url[:50]}... {str(e)[:50]}")
                    await self.log_failed_url(url, f"Parse error: {str(e)}")
                
                # Free memory
                del html
                gc.collect()
                
            except asyncio.TimeoutError:
                continue
            except Exception as e:
                print(f"✗ Worker error: {str(e)}")
                break
    
    async def zip_batch_async(self, batch_num: int):
        """Async zip operation."""
        loop = asyncio.get_event_loop()
        await loop.run_in_executor(None, self._zip_batch_sync, batch_num)
    
    def _zip_batch_sync(self, batch_num: int):
        """Synchronous zip operation."""
        html_files = sorted(glob.glob(str(self.html_dir / "*.html")))
        
        if not html_files:
            return
        
        zip_filename = self.html_dir / f"batch_{batch_num:03d}.zip"
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for html_file in html_files:
                zipf.write(html_file, arcname=Path(html_file).name)
        
        # Delete original HTML files
        for html_file in html_files:
            os.remove(html_file)
        
        print(f"📦 Zipped {len(html_files)} files to {zip_filename.name}")
    
    async def log_failed_url(self, url: str, error: str):
        """Log failed URLs."""
        self.failed_counter += 1
        with open(self.failed_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps({
                'url': url,
                'error': error,
                'timestamp': datetime.now().isoformat()
            }, ensure_ascii=False) + '\n')
    
    async def run(self, urls: List[str]):
        """Main async runner with concurrent scraping & parsing."""
        print(f"🚀 Starting optimized scraper: {len(urls)} URLs")
        print(f"   Concurrent scrapers: {self.max_concurrent}")
        print(f"   Parse workers: {self.parse_workers}")
        print(f"   Output: {self.results_file}\n")
        
        # Create semaphore to limit concurrent scrapes
        semaphore = asyncio.Semaphore(self.max_concurrent)
        
        # Start parse workers
        parse_tasks = [
            asyncio.create_task(self.parse_worker())
            for _ in range(self.parse_workers)
        ]
        
        # Create scrape tasks
        scrape_tasks = [
            asyncio.create_task(self.scrape_with_delay(url, semaphore))
            for url in urls
        ]
        
        # Wait for all scrapes to complete
        scrape_results = await asyncio.gather(*scrape_tasks, return_exceptions=True)
        
        # Put results in parse queue
        for result in scrape_results:
            if result:
                await self.scrape_queue.put(result)
        
        # Signal workers to stop
        for _ in range(self.parse_workers):
            await self.scrape_queue.put(None)
        
        # Wait for all parsing to complete
        await asyncio.gather(*parse_tasks)
        
        # Final zip of remaining files
        remaining_html = glob.glob(str(self.html_dir / "*.html"))
        if remaining_html:
            await self.zip_batch_async(self.batch_counter)
        
        # Summary
        print(f"\n✅ Complete!")
        print(f"   ✓ Parsed: {self.parsed_counter}")
        print(f"   ✗ Failed: {self.failed_counter}")
        print(f"   📝 JSONL: {self.results_file}")
        print(f"   ❌ Failed URLs: {self.failed_file}")


# ============ USAGE ============
urls = dff['URL'].tolist()

scraper = G1ArticleScraperOptimized(
    max_concurrent=2,     # 2 concurrent scraping tasks
    parse_workers=2,      # 2 parsing workers
    min_wait=6.0,
    max_wait=7.0,
)

await scraper.run(urls[:10])  # Test with first 10

🚀 Starting optimized scraper: 10 URLs
   Concurrent scrapers: 2
   Parse workers: 2
   Output: data/scraped_articles/articles.jsonl

✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1202885 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1260334 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1221100 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1378352 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1364514 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1393376 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1255228 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1650255 chars)
✓ Scraped & saved: https://g1.globo.com/fato-ou-fake/noticia/2026/03/... (1251930 chars)
✗ Scrape failed: https://g1.globo.com/fato-ou-fake/noticia/2026/03